# 00 · 信号处理预备知识

本笔记是阅读 `01_SSVEP信号与预处理` 之前的**预备知识**。
如果你对以下概念已经熟悉，可以跳过本笔记：

- 采样率 `fs`、采样点数 `n`、采样间隔 `dt`
- 时域 / 频域、直流分量（均值）
- FFT（快速傅里叶变换）、频率分辨率 `fs/n`
- 频谱泄漏、窗函数（Hann 窗）
- 幅度谱 vs 功率谱

建议配合 `01_SSVEP信号与预处理.ipynb` 一起阅读。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 一、采样：把连续信号变成数字序列

现实世界的信号（声音、脑电）是**连续**的，但电脑只能处理**离散**的数字。
采样就是每隔一小段时间取一个值，得到一串数字。

三个核心概念：

| 符号 | 含义 | 单位 | 例子 |
|------|------|------|------|
| `fs` | **采样率**：每秒采多少个点 | Hz（次/秒） | 250 Hz |
| `n`  | **采样点数**：总共采了多少个点 | 个 | 750 |
| `dt` | **采样间隔**：相邻两个点的时间差 | 秒 | `1/fs = 0.004 s` |
| `T`  | **总时长** | 秒 | `n/fs = n*dt = 3 s` |

### 它们之间的关系

$$dt = \frac{1}{fs}, \quad T = n \cdot dt = \frac{n}{fs}$$

**采样率越高，时间精度越高，能捕捉到的高频成分越多。**
根据**奈奎斯特采样定理**：采样率 `fs` 只能正确还原 **`fs/2` 以下**的频率成分。
例如 fs=250 Hz 时，最高只能分析到 125 Hz。

In [ ]:
fs = 250          # 每秒 250 个点
duration = 3.0    # 持续 3 秒
n = int(fs * duration)   # 总点数 = 750
dt = 1 / fs       # 采样间隔 = 0.004 秒

print(f'采样率 fs   = {fs} Hz')
print(f'总时长 T    = {duration} s')
print(f'采样点数 n  = {n}')
print(f'采样间隔 dt = {dt} s = {dt*1000:.1f} ms')

# 用 np.arange 生成时间轴：0, dt, 2dt, ..., 不包含 duration
t = np.arange(n) * dt
print(f'时间轴前 5 个点: {t[:5]}')

## 二、时域 vs 频域：从两个角度看信号

### 时域（Time Domain）
- 横轴是**时间**，纵轴是信号**幅度**。
- 看到的是信号随时间的起伏。
- 例如：一段正弦波 `sin(2π·10·t)` 在时域是一条周期起伏的曲线。

### 频域（Frequency Domain）
- 横轴是**频率**，纵轴是该频率成分的**强度**。
- 任何周期信号都可以分解成若干个不同频率的正弦波之和。
- 10 Hz 正弦波在频域就是 10 Hz 处的一个尖峰。

### 为什么需要频域？
SSVEP 信号里混杂了基频、谐波、噪声，在时域上看起来很乱，
但在频域上，基频和谐波会变成**清晰的尖峰**，一眼就能看出来。
这就是为什么 SSVEP 分类主要看频谱。

In [ ]:
f = 10.0  # 10 Hz 的正弦波
x = np.sin(2 * np.pi * f * t)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# 时域图
axes[0].plot(t, x, color='steelblue')
axes[0].set_xlim(0, 0.5)
axes[0].set(xlabel='时间 (s)', ylabel='幅度', title='时域：10 Hz 正弦波')

# 频域图（先简单用 FFT，下面会详细讲）
sp = np.abs(np.fft.rfft(x))**2
fr = np.fft.rfftfreq(n, dt)
axes[1].plot(fr, sp, color='darkorange')
axes[1].set_xlim(0, 30)
axes[1].set(xlabel='频率 (Hz)', ylabel='功率', title='频域：10 Hz 处一个尖峰')

plt.tight_layout()
plt.show()

## 三、直流分量（DC）：信号的平均值

信号的**平均值**在频域对应 **0 Hz**（直流，Direct Current）。

### 问题：如果信号均值不为零，会发生什么？
- 0 Hz 处会出现一个**非常高的尖峰**。
- 这个峰的能量往往远大于其他频率，导致画图时纵轴被压缩，
  你关心的 10 Hz、20 Hz 峰变得几乎看不见。

### 解决方法：减去均值

```python
x = x - x.mean()
```

这就是 `spectrum` 函数第一行代码的作用。减去均值后，信号围绕 0 上下波动，0 Hz 处的尖峰消失，其他频率的峰就能看清楚了。

In [ ]:
# 构造一个均值不为零的信号：10 Hz 正弦波 + 直流偏置 0.5
x_dc = np.sin(2 * np.pi * 10 * t) + 0.5
print(f'去均值前的平均值: {x_dc.mean():.4f}')

x_centered = x_dc - x_dc.mean()
print(f'去均值后的平均值: {x_centered.mean():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# 去均值前：0 Hz 处有大尖峰
sp_before = np.abs(np.fft.rfft(x_dc))**2
fr = np.fft.rfftfreq(n, dt)
axes[0].plot(fr, sp_before, color='gray')
axes[0].set_xlim(0, 30)
axes[0].set_title('去均值前：0 Hz 处大尖峰压扁了纵轴')
axes[0].set_xlabel('频率 (Hz)')

# 去均值后：0 Hz 峰消失，10 Hz 峰清晰可见
sp_after = np.abs(np.fft.rfft(x_centered))**2
axes[1].plot(fr, sp_after, color='darkorange')
axes[1].set_xlim(0, 30)
axes[1].set_title('去均值后：10 Hz 峰清晰可见')
axes[1].set_xlabel('频率 (Hz)')

plt.tight_layout()
plt.show()

## 四、FFT：把信号从时域变到频域

**傅里叶变换**（Fourier Transform）是连接时域和频域的桥梁。
**FFT**（Fast Fourier Transform，快速傅里叶变换）是它的高效算法实现。

### 核心思想
任何信号都可以看成是**一堆不同频率的正弦波叠加**而成的。
FFT 就是算出：这个信号里，每个频率的正弦波**贡献了多少能量**。

### 实数信号的 FFT：`rfft`
脑电信号是**实数信号**（每个采样点就是一个实数）。
实数信号的频谱有一个性质：**负频率部分是正频率的镜像**（共轭对称）。
所以负频率完全是冗余的，不需要算。

- `np.fft.fft(x)` —— 返回完整的 n 个点（含正负频率），有冗余。
- `np.fft.rfft(x)` —— 只返回**非负频率**部分（0 ~ fs/2），长度 `n//2 + 1`，更高效。

### 频率轴：`rfftfreq`
FFT 的输出只是一串复数，你不知道每个点对应哪个频率。
`np.fft.rfftfreq(n, 1/fs)` 会算出每个频率点的真实频率值。

### 频率分辨率
相邻两个频率点之间的间隔是：

$$\Delta f = \frac{fs}{n} = \frac{1}{T}$$

本例中 `Δf = 250/750 ≈ 0.333 Hz`。

**想提高频率分辨率？两种办法：**
1. 增加采样点数 `n`（采集更长时间）。
2. 降低采样率 `fs`（但会牺牲最高可分析频率）。

实际上，频率分辨率只取决于**总时长 T**：`Δf = 1/T`。

In [ ]:
# 对比 fft 和 rfft 的长度
x = np.sin(2 * np.pi * 10 * t)

full_fft = np.fft.fft(x)
real_fft = np.fft.rfft(x)

print(f'信号长度 n       = {n}')
print(f'fft 输出长度      = {len(full_fft)}  (含正负频率)')
print(f'rfft 输出长度     = {len(real_fft)}  (只含非负频率)')
print(f'n//2 + 1         = {n//2 + 1}')

# 频率轴
fr = np.fft.rfftfreq(n, 1/fs)
print(f'频率分辨率 Δf     = {fr[1] - fr[0]:.4f} Hz')
print(f'频率轴范围        = {fr[0]:.1f} ~ {fr[-1]:.1f} Hz')
print(f'奈奎斯特频率 fs/2 = {fs/2:.1f} Hz')

## 五、幅度谱 vs 功率谱

FFT 的输出是**复数**，不能直接画图。我们通常取它的模：

- `np.abs(rfft(x))` → **幅度谱**：每个频率成分的振幅大小。
- `np.abs(rfft(x))**2` → **功率谱**：幅度的平方，代表该频率的能量。

为什么用平方？因为**功率 = 幅度²**，平方后强弱对比更明显，
而且功率有明确的物理意义（能量）。`spectrum` 函数用的就是功率谱。

In [ ]:
x = np.sin(2 * np.pi * 10 * t) + 0.5 * np.sin(2 * np.pi * 20 * t)
X = np.fft.rfft(x)
fr = np.fft.rfftfreq(n, 1/fs)

amp = np.abs(X)       # 幅度谱
power = amp ** 2      # 功率谱

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(fr, amp, color='steelblue')
axes[0].set_xlim(0, 30)
axes[0].set_title('幅度谱 abs(X)')
axes[0].set_xlabel('频率 (Hz)')

axes[1].plot(fr, power, color='darkorange')
axes[1].set_xlim(0, 30)
axes[1].set_title('功率谱 abs(X)²（强弱对比更明显）')
axes[1].set_xlabel('频率 (Hz)')

plt.tight_layout()
plt.show()

## 六、频谱泄漏：为什么有限长度信号有问题

FFT 默认假设信号是**无限周期重复**的。
但实际上我们只有一段有限长的信号。
当把这段信号首尾拼起来时，**边界处往往不连续**，产生一个突变。

这个突变会在频谱上产生**频谱泄漏**（Spectral Leakage）：
- 原本集中在某个频率的能量，扩散到了相邻频率。
- 尖峰变宽，还可能出现伪峰。

### 演示
一个严格 10 Hz 的信号，如果总时长不是 10 Hz 的整数倍周期，
首尾拼接就不连续，频谱会出现泄漏。

In [ ]:
fs = 250

# 情况 1：时长正好是 10 Hz 的整数倍周期（0.5 s = 5 个周期），无泄漏
n_good = int(fs * 0.5)          # 125 个点
t_good = np.arange(n_good) / fs
x_good = np.sin(2 * np.pi * 10 * t_good)

# 情况 2：时长不是整数倍周期（0.55 s），首尾不连续，有泄漏
n_bad = int(fs * 0.55)          # 137 个点
t_bad = np.arange(n_bad) / fs
x_bad = np.sin(2 * np.pi * 10 * t_bad)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

sp_good = np.abs(np.fft.rfft(x_good))**2
fr_good = np.fft.rfftfreq(n_good, 1/fs)
axes[0].plot(fr_good, sp_good, color='steelblue')
axes[0].set_xlim(0, 30)
axes[0].set_title('整数倍周期：尖峰干净，无泄漏')
axes[0].set_xlabel('频率 (Hz)')

sp_bad = np.abs(np.fft.rfft(x_bad))**2
fr_bad = np.fft.rfftfreq(n_bad, 1/fs)
axes[1].plot(fr_bad, sp_bad, color='crimson')
axes[1].set_xlim(0, 30)
axes[1].set_title('非整数倍周期：能量扩散，频谱泄漏')
axes[1].set_xlabel('频率 (Hz)')

plt.tight_layout()
plt.show()

## 七、窗函数：给泄漏"打补丁"

实际中我们几乎不可能保证信号时长是每个频率的整数倍周期。
**窗函数**（Window）就是用来减轻频谱泄漏的标准工具。

### 做法
把信号乘以一个窗函数，让信号两端平滑地过渡到 0，
这样首尾拼接时就不会有突变。

### Hann 窗（汉宁窗）
形状像一个钟形曲线：中间为 1，两端为 0。

```python
window = np.hanning(n)
windowed = x * window
```

### 代价
- 优点：大大减轻频谱泄漏，尖峰更干净。
- 缺点：主瓣（尖峰的宽度）会稍微变宽，频率分辨率略有下降。

对 SSVEP 这种有清晰谐波峰的信号，加窗**利远大于弊**。

In [ ]:
# Hann 窗的形状
window = np.hanning(n)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(t, window, color='green')
axes[0].set_title('Hann 窗：中间 1，两端 0')
axes[0].set_xlabel('时间 (s)')

# 不加窗 vs 加窗 对比（用非整数倍周期的信号）
x = np.sin(2 * np.pi * 10 * t_bad)
x_windowed = x * np.hanning(n_bad)

sp_raw = np.abs(np.fft.rfft(x))**2
sp_win = np.abs(np.fft.rfft(x_windowed))**2
fr = np.fft.rfftfreq(n_bad, 1/fs)

axes[1].plot(fr, sp_raw / sp_raw.max(), label='不加窗', color='crimson', alpha=0.7)
axes[1].plot(fr, sp_win / sp_win.max(), label='加 Hann 窗', color='green', alpha=0.8)
axes[1].set_xlim(5, 15)
axes[1].set_title('加窗前后对比（归一化）')
axes[1].set_xlabel('频率 (Hz)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 八、串起来：`spectrum` 函数逐行回顾

现在再看 `01_SSVEP信号与预处理` 里的 `spectrum` 函数，每一行都有意义：

```python
def spectrum(x, fs):
    x = x - x.mean()                    # ① 去直流：消除 0 Hz 偏置
    n = len(x)                          # ② 取采样点数
    windowed = x * np.hanning(n)        # ③ 加窗：减轻频谱泄漏
    sp = np.abs(np.fft.rfft(windowed))**2   # ④ 实信号 FFT，取功率谱
    fr = np.fft.rfftfreq(n, 1/fs)          # ⑤ 生成频率轴
    return fr, sp                        # ⑥ 返回 (频率, 功率)
```

| 步骤 | 解决的问题 |
|------|-----------|
| ① 去均值 | 0 Hz 直流分量压扁纵轴 |
| ③ 加 Hann 窗 | 有限截断导致的频谱泄漏 |
| ④ rfft + 平方 | 把时域信号变成可观察的频域功率 |
| ⑤ rfftfreq | 给每个频率点标上真实 Hz 值 |

这是数字信号处理里**频谱分析的标准范式**，不只是 SSVEP，几乎所有周期信号分析都用这个流程。

In [ ]:
# 完整复刻 spectrum 函数并验证
def spectrum(x, fs):
    x = x - x.mean()
    n = len(x)
    windowed = x * np.hanning(n)
    sp = np.abs(np.fft.rfft(windowed))**2
    fr = np.fft.rfftfreq(n, 1/fs)
    return fr, sp

# 构造一个 10 Hz 基频 + 20 Hz 谐波的信号
x = np.sin(2*np.pi*10*t) + 0.5*np.sin(2*np.pi*20*t + 0.3)
fr, sp = spectrum(x, fs)

band = (fr >= 1) & (fr <= 45)
plt.figure(figsize=(9, 3.5))
plt.plot(fr[band], sp[band], color='darkorange')
for h in (1, 2):
    plt.axvline(h*10, color='gray', ls='--', lw=0.8)
    plt.text(h*10 + 0.3, sp[band].max()*0.9, f'{h}f')
plt.xlabel('频率 (Hz)')
plt.ylabel('功率')
plt.title('完整 spectrum 函数输出：10 Hz 和 20 Hz 两个尖峰')
plt.tight_layout()
plt.show()

## 九、小结

本笔记的关键概念：

1. **采样**：`fs`（采样率）、`n`（点数）、`dt=1/fs`（间隔）、`T=n/fs`（时长）。
2. **时域 vs 频域**：时域看波形起伏，频域看各频率成分的强度。
3. **直流分量**：信号均值，对应 0 Hz，去均值可避免纵轴被压扁。
4. **FFT**：`rfft` 只算非负频率；`rfftfreq` 生成频率轴；频率分辨率 `Δf = fs/n = 1/T`。
5. **幅度谱 vs 功率谱**：`abs(X)` 和 `abs(X)²`，后者强弱对比更明显。
6. **频谱泄漏**：有限信号首尾不连续导致能量扩散。
7. **窗函数**：Hann 窗让信号两端平滑过渡到 0，减轻泄漏。

掌握这些后，再去看 `01_SSVEP信号与预处理.ipynb` 里的 `spectrum` 函数和各种频谱图，
每一步的意图都会很清楚。